# RAG on Azure — Day 6: Source-Cited & Verified Answers 📚

**Use case:** every answer is traceable. The user can see *which document* and *which section* supports each claim, and a second AI pass verifies that the cited passage actually supports what the answer says.

**What's new vs Day 5:**
- **Richer source metadata** — each chunk now carries `source`, `page`, and `section` (extracted from the document's heading structure), not just the filename.
- **Citation parsing** — we parse the `[N]` markers from the answer and know which retrieved passages were actually used (versus just retrieved).
- **Formatted source list** — under each answer, a clean "Sources used" block listing only the cited chunks with their section names.
- **Citation verification** — an LLM-as-judge step that checks, per citation, whether the cited passage *actually* supports the claim in the answer. Hallucinated or weakly-supported cites get flagged.

**Done when:** every answer prints with a sources list, each citation is labelled `supported` / `partial` / `unsupported`, and you can see at a glance which claims are auditable.

**Data:** reuses the Day 2 handbook PDFs from `data/`. We re-ingest them with section-aware chunking into a new index (`rag-handbook-day6`).

## Prerequisites
Day 2 handbook PDFs in your `data/` folder (the ones with numbered sections like "1. Purpose", "2. Annual PTO Allowance", etc.).

## 0. Install dependencies

In [ ]:
%pip install -q "openai>=1.55.3" "azure-search-documents>=11.5.1" pypdf python-dotenv

## 1. Load configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

explicit = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env")
env_path = explicit if explicit.exists() else (Path(find_dotenv()) if find_dotenv() else None)

if env_path and Path(env_path).exists():
    for line in Path(env_path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
    load_dotenv(env_path, override=False)
else:
    raise FileNotFoundError("No .env file found.")

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_SEARCH_ENDPOINT    = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY     = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME               = os.getenv("AZURE_SEARCH_INDEX_DAY6", "rag-handbook-day6")

for _n, _v in [("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
               ("AZURE_OPENAI_API_KEY",  AZURE_OPENAI_API_KEY),
               ("AZURE_SEARCH_ENDPOINT", AZURE_SEARCH_ENDPOINT),
               ("AZURE_SEARCH_API_KEY",  AZURE_SEARCH_API_KEY)]:
    assert _v, f"Set {_n} in your .env"

print("Config OK. Using index:", INDEX_NAME)

## 2. Clients

In [ ]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

aoai = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)
search_cred = AzureKeyCredential(AZURE_SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=search_cred)
print("Clients ready.")

## 3. Load PDFs with section-aware extraction

The new bit: we split each PDF into **sections** by detecting numbered headings like `1. Purpose`, `2. Annual PTO Allowance`, etc. Each section becomes its own chunk (or chunks, if long), and the **section title** travels into the index so citations can say *"01_PTO_Policy.pdf — Annual PTO Allowance"* instead of just the filename.

In [ ]:
import re
from pypdf import PdfReader

DATA_DIR = Path("data")

def extract_sections(text):
    """Split text into (section_title, body) pairs.
    Detects numbered heading lines like '1. Title', '2. Title' as section starts."""
    sections = []
    current_title = "Introduction"
    current_body = []
    for line in text.splitlines():
        stripped = line.strip()
        m = re.match(r"^(\d+)\.\s+(.+)$", stripped)
        if m and len(m.group(2)) < 80:                 # title-like length
            if current_body:
                sections.append((current_title, "\n".join(current_body).strip()))
            current_title = m.group(2)
            current_body = []
        else:
            current_body.append(line)
    if current_body:
        sections.append((current_title, "\n".join(current_body).strip()))
    return [(t, b) for t, b in sections if b.strip()]

documents = []     # list of dicts: {source, page, section, text}
pdfs = sorted(DATA_DIR.glob("*.pdf")) if DATA_DIR.exists() else []
print("Looking in:", DATA_DIR.resolve())
assert pdfs, f"No PDFs found in {DATA_DIR.resolve()}"

for path in pdfs:
    reader = PdfReader(str(path))
    for page_num, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        for section_title, body in extract_sections(page_text):
            documents.append({
                "source":  path.name,
                "page":    page_num,
                "section": section_title,
                "text":    body,
            })

print(f"\nExtracted {len(documents)} sections from {len(pdfs)} PDFs:")
for d in documents[:8]:
    print(f"  {d['source']:<35} p{d['page']}  [{d['section']}]")
if len(documents) > 8:
    print(f"  ... and {len(documents) - 8} more")

## 4. Chunk and carry metadata through

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=150):
    text = text.strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            window = text[start:end]
            brk = max(window.rfind("\n"), window.rfind(". "), window.rfind(" "))
            if brk > chunk_size * 0.5:
                end = start + brk + 1
        chunks.append(text[start:end].strip())
        start = max(0, end - overlap)
    return [c for c in chunks if c]

records = []
for d in documents:
    safe_src = re.sub(r"[^A-Za-z0-9_-]", "_", d["source"])
    safe_sec = re.sub(r"[^A-Za-z0-9_-]", "_", d["section"])[:30]
    for i, chunk in enumerate(chunk_text(d["text"])):
        records.append({
            "id":      f"{safe_src}-p{d['page']}-{safe_sec}-{i}",
            "content": chunk,
            "source":  d["source"],
            "page":    d["page"],
            "section": d["section"],
        })

print(f"{len(records)} chunks. Example:")
print(records[0])

## 5. Embed the chunks

In [ ]:
def embed_texts(texts, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        resp = aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=texts[i:i+batch_size])
        out.extend(d.embedding for d in resp.data)
    return out

def embed(text):
    return embed_texts([text])[0]

chunk_vectors = embed_texts([r["content"] for r in records])
EMBED_DIM = len(chunk_vectors[0])
print(f"Embedded {len(chunk_vectors)} chunks. Dimension: {EMBED_DIM}")

## 6. Create index with `page` and `section` fields

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

fields = [
    SimpleField(name="id",      type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="source",  type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="page",    type=SearchFieldDataType.Int32,  filterable=True, sortable=True),
    SimpleField(name="section", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index to delete (ok):", type(e).__name__)

index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search))
print(f"Index '{INDEX_NAME}' created with section + page metadata.")

## 7. Upload

In [ ]:
search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_cred)

docs = [{**r, "contentVector": v} for r, v in zip(records, chunk_vectors)]
for i in range(0, len(docs), 100):
    search_client.upload_documents(documents=docs[i:i+100])
print(f"Uploaded {len(docs)} chunks.")

## 8. Retrieve — returns rich metadata

In [ ]:
from azure.search.documents.models import VectorizedQuery

def retrieve(query, k=5):
    vq = VectorizedQuery(vector=embed(query), k_nearest_neighbors=k, fields="contentVector")
    results = search_client.search(
        search_text=None, vector_queries=[vq],
        select=["content", "source", "page", "section"],
    )
    return [
        {"content": r["content"], "source": r["source"], "page": r["page"],
         "section": r["section"], "score": r["@search.score"]}
        for r in results
    ]

# Sanity check
for h in retrieve("How many PTO days?", k=3):
    print(f"[{h['score']:.3f}] {h['source']} p{h['page']} — {h['section']}")

## 9. Generate an answer with strict citation rules

We force the model to cite passage numbers like `[1]`, `[2]` for every claim. Day 6's verification step (next section) will inspect *which* citations were used and check each one.

In [ ]:
ANSWER_SYSTEM = (
    "You answer questions strictly from the numbered context passages provided. "
    "Cite passage numbers like [1], [2] inline next to every claim you make. "
    "If the answer is not in the context, say you don't know based on the documents. "
    "Do NOT cite passages you didn't actually use."
)

def answer_question(question, k=5):
    hits = retrieve(question, k=k)
    context = "\n\n".join(
        f"[{i+1}] (source: {h['source']}, page {h['page']}, section: {h['section']})\n{h['content']}"
        for i, h in enumerate(hits)
    )
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), hits

## 10. The Day 6 work — parse + verify citations

Two small functions:

1. **`parse_citations`** — finds every `[N]` marker in the answer, mapped to the *sentence* it appears in. We verify each cite against the specific sentence it supports (more precise than verifying against the whole answer).
2. **`verify_citation`** — an LLM-as-judge call: "Does this passage support this claim sentence? Answer SUPPORTED, PARTIAL, or UNSUPPORTED."

In [ ]:
CITE_RE = re.compile(r"\[(\d+)\]")

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

def parse_citations(answer, hits):
    """Return list of {cite, sentence, passage} for every [N] in the answer."""
    found = []
    for sent in split_sentences(answer):
        for m in CITE_RE.finditer(sent):
            n = int(m.group(1))
            if 1 <= n <= len(hits):
                found.append({"cite": n, "sentence": sent, "passage": hits[n - 1]})
    return found

VERIFY_SYSTEM = (
    "You are a strict citation verifier. Decide whether the source passage SUPPORTS the claim sentence. "
    "Respond with exactly one word: SUPPORTED, PARTIAL, or UNSUPPORTED. "
    "SUPPORTED = the passage clearly states the claim. "
    "PARTIAL = the passage is related but does not fully state the claim. "
    "UNSUPPORTED = the passage does not support the claim."
)

def verify_citation(claim_sentence, passage_text):
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": VERIFY_SYSTEM},
            {"role": "user",   "content": f"Claim sentence: {claim_sentence}\n\nSource passage: {passage_text}\n\nVerdict:"},
        ],
        temperature=0,
        max_tokens=8,
    )
    verdict = resp.choices[0].message.content.strip().upper().split()[0]
    return verdict if verdict in ("SUPPORTED", "PARTIAL", "UNSUPPORTED") else "PARTIAL"

def verify_all(answer, hits):
    items = parse_citations(answer, hits)
    for it in items:
        it["verdict"] = verify_citation(it["sentence"], it["passage"]["content"])
    return items

## 11. Format the answer with a verified sources block

In [ ]:
VERDICT_TAG = {
    "SUPPORTED":   "[supported]",
    "PARTIAL":     "[partial]   ",
    "UNSUPPORTED": "[UNSUPPORTED]",
}

def format_answer(question, answer, verifications, hits):
    print("=" * 78)
    print(f"Q: {question}\n")
    print("A:", answer)
    print()

    # Unique cited passages
    cited = {}
    for v in verifications:
        n = v["cite"]
        if n not in cited:
            cited[n] = {"passage": v["passage"], "worst": v["verdict"]}
        else:
            # If any verification of this cite is UNSUPPORTED, mark it as such
            order = {"SUPPORTED": 0, "PARTIAL": 1, "UNSUPPORTED": 2}
            if order[v["verdict"]] > order[cited[n]["worst"]]:
                cited[n]["worst"] = v["verdict"]

    print("Sources used:")
    if not cited:
        print("  (none — the answer made no citations)")
    for n in sorted(cited):
        p = cited[n]["passage"]
        tag = VERDICT_TAG[cited[n]["worst"]]
        print(f"  [{n}] {tag}  {p['source']} (p{p['page']}) — {p['section']}")

    # Detailed per-claim verifications
    if verifications:
        print("\nVerification per claim:")
        for v in verifications:
            tag = VERDICT_TAG[v["verdict"]]
            sent = v["sentence"][:90] + ("..." if len(v["sentence"]) > 90 else "")
            print(f"  [{v['cite']}] {tag}  \"{sent}\"")
    print()

## 12. End-to-end demo

In [ ]:
def ask(question):
    answer, hits = answer_question(question)
    verifications = verify_all(answer, hits)
    format_answer(question, answer, verifications, hits)

ask("How many PTO days do full-time employees get?")
ask("When do new hires start accruing PTO?")
ask("What are the rules for working remotely?")

## 13. A stress test — a question that invites hallucinated cites

Ask something the documents *partially* address. The model may be tempted to cite passages that are only tangentially related. Verification catches it.

In [ ]:
ask("What is the maternity leave policy?")   # not in the docs - the verifier should flag any cites as UNSUPPORTED

## ✅ Day 6 checklist

- [ ] Each chunk carries `source`, `page`, and `section`
- [ ] Citations are parsed from the answer's `[N]` markers
- [ ] A clean "Sources used" block prints under every answer, with section names
- [ ] Each citation is verified by an LLM judge: `SUPPORTED`, `PARTIAL`, or `UNSUPPORTED`
- [ ] On a deliberately off-topic question, at least one citation is flagged as `PARTIAL` or `UNSUPPORTED`

### What you learned today
- **Citations are a UX feature**, not just an internal label. They turn opaque answers into auditable ones.
- **Section-level provenance** is far more useful than filename-only. Pointing to "Remote Work Policy — Core Hours" beats pointing to a 10-page PDF.
- **Verification is a second LLM pass.** Costlier per query, but it catches a class of failure (over-citation, weakly-grounded claims) that a single generation pass can't.

### Next up — Day 7: consolidation + GitHub
Refactor Days 2–6 into a clean monorepo with shared utilities, per-day READMEs, an `.env.example`, and a top-level results summary. The first portfolio-worthy commit.